# 📊 Notebook B5 — Path B: Evaluate PolyWhisper MoLE & Video Ad Placement

**Path B Concrete Specs — Stages 5 & 6: Evaluation & Deployment**

This notebook evaluates the **Path B PolyWhisper MoLE Architecture**:
1. **Baseline**: Base Whisper (`SayedShaun/bengali-whisper-medium`)
2. **Path B PolyWhisper MoLE**: Bengali Expert + English Expert + MoLE Router MLP

### Evaluation Protocol:
- **Held-out code-switched validation set** (`val_router.csv` from `data_path_b/`)
- **NFC Unicode normalization** on both hypothesis and ground truth
- **Metrics**:
  1. **WER** (Word Error Rate via `jiwer` — lower is better)
  2. **EWER** (English Word Emission Rate — higher is better): Did English words survive in Latin script or were they transliterated into Bengali?
  3. **Router Allocation**: % of tokens routed to Bengali expert vs English expert
- **Real Video Inference**: Tests on your actual Hoichoi `.mp4` video clips
- **Contextual Ad Matching**: Automatically matches detected video scenes with `brands.json` for contextual ad insertion.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Mount Drive & Model Availability Check
# Main Google Drive Folder: https://drive.google.com/drive/folders/1zqBtJvJ8ZwZNNpLCiS2F-x-yBpMxYXfE?usp=sharing
# data_path_b Folder: https://drive.google.com/drive/folders/1hbAfVrgYEJpASZGxQmNX0zaEok3ry19D?usp=sharing
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
DRIVE_ROOT      = '/content/drive/MyDrive/hoichoi'
DATA_DIR        = DRIVE_ROOT + '/data_path_b'
BN_EXPERT       = DRIVE_ROOT + '/adapters_path_b/bengali_expert/best'
EN_EXPERT       = DRIVE_ROOT + '/adapters_path_b/english_expert/best'
ROUTER_WEIGHTS  = DRIVE_ROOT + '/router_path_b/best/router.pt'
DATA_B_DIR_ID   = '1hbAfVrgYEJpASZGxQmNX0zaEok3ry19D'

# Direct validation dataset path in data_path_b
VAL_CSV = DATA_DIR + '/val_router.csv'
if not os.path.exists(VAL_CSV) and os.path.exists(DATA_DIR + '/val_bengali_expert.csv'):
    VAL_CSV = DATA_DIR + '/val_bengali_expert.csv'

if not os.path.exists(VAL_CSV):
    print(f'📥 Fetching data_path_b into {DATA_DIR} from Drive folder link...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    try:
        gdown.download_folder(id=DATA_B_DIR_ID, output=DATA_DIR, quiet=False, use_cookies=False)
    except Exception as e:
        print('Download notice:', e)
    if os.path.exists(DATA_DIR + '/val_router.csv'):
        VAL_CSV = DATA_DIR + '/val_router.csv'

print('=' * 65)
print('  PATH B MODEL READINESS FOR EVALUATION')
print('=' * 65)
print(f'  Val Dataset       : {"✅ Found: " + VAL_CSV if os.path.exists(VAL_CSV) else "❌ Missing"}')
print(f'  Path B BN Expert  : {"✅ Found: " + BN_EXPERT if os.path.exists(BN_EXPERT + "/adapter_config.json") else "❌ Missing (Run Notebook B2)"}')
print(f'  Path B EN Expert  : {"✅ Found: " + EN_EXPERT if os.path.exists(EN_EXPERT + "/adapter_config.json") else "❌ Missing (Run Notebook B3)"}')
print(f'  Path B Router MLP : {"✅ Found: " + ROUTER_WEIGHTS if os.path.exists(ROUTER_WEIGHTS) else "❌ Missing (Run Notebook B4)"}')
print('=' * 65)

assert os.path.exists(VAL_CSV), f'❌ Validation CSV not found at {VAL_CSV}! Run Notebook B1 first.'
if not (os.path.exists(BN_EXPERT + '/adapter_config.json') and os.path.exists(EN_EXPERT + '/adapter_config.json')):
    print('⚠️ Notice: Path B expert adapters not found yet. Please ensure Notebooks B2 and B3 have completed.')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Dependencies & Environment Setup
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, sys
print('Installing evaluation dependencies (transformers, peft, jiwer, librosa, soundfile) ...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40.0', 'peft>=0.10.0,<0.14.0', 'accelerate>=0.29.0',
    'jiwer', 'librosa', 'soundfile'])

# Bypass Colab torchao/peft probe incompatibility
try:
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda: False
except Exception:
    pass

# Install ffmpeg for video audio extraction
subprocess.run(['apt-get', 'install', '-qq', 'ffmpeg'], capture_output=True)
print('✅ Environment & ffmpeg ready.')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running evaluation on: {device}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Evaluation Metrics (NFC, WER, EWER)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import unicodedata
from dataclasses import dataclass
from typing import List
from jiwer import wer

BENGALI_RANGE = (0x0980, 0x09FF)
LATIN_RANGE   = (0x0041, 0x007A)

def nfc(text: str) -> str:
    """NFC-normalize and clean extra whitespace."""
    return ' '.join(unicodedata.normalize('NFC', str(text)).split())

def is_bengali_word(word: str) -> bool:
    return any(BENGALI_RANGE[0] <= ord(c) <= BENGALI_RANGE[1] for c in word)

def is_latin_word(word: str) -> bool:
    return any(LATIN_RANGE[0] <= ord(c.lower()) <= LATIN_RANGE[1] for c in word)

@dataclass
class EWERResult:
    ref_english_words: int = 0
    emitted_correctly: int = 0
    transliterated:    int = 0
    missing:           int = 0
    ewer:              float = 0.0

def compute_ewer(reference: str, hypothesis: str) -> EWERResult:
    ref_words = nfc(reference).split()
    hyp_words = nfc(hypothesis).split()
    
    result = EWERResult()
    hyp_idx = 0
    
    for ref_word in ref_words:
        if not is_latin_word(ref_word): continue
        result.ref_english_words += 1
        
        found = False
        for j in range(hyp_idx, min(hyp_idx + 6, len(hyp_words))):
            if hyp_words[j].lower() == ref_word.lower():
                result.emitted_correctly += 1
                hyp_idx = j + 1
                found = True
                break
            elif is_bengali_word(hyp_words[j]) and not is_latin_word(hyp_words[j]):
                result.transliterated += 1
                hyp_idx = j + 1
                found = True
                break
        if not found:
            result.missing += 1
            
    if result.ref_english_words > 0:
        result.ewer = result.emitted_correctly / result.ref_english_words
    return result

print('✅ NFC, WER, and EWER metric functions initialized')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — PolyWhisper MoLE Architecture & Inference Engine
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, torch, librosa
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel

class WhisperMoERouter(nn.Module):
    def __init__(self, d_model=1024, hidden_dim=32, num_experts=2, dropout=0.1):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_experts)
        )
    def forward(self, x):
        return self.mlp(x)

def resolve_audio_path(apath):
    """Resolves local Drive or content audio path."""
    if os.path.exists(apath):
        return apath
    fname = os.path.basename(apath)
    for cdir in [
        '/content/drive/MyDrive/hoichoi/mucs_raw/train',
        '/content/drive/MyDrive/hoichoi/mucs_raw',
        '/content/drive/MyDrive/mucs_raw/train',
        '/content/drive/MyDrive/mucs_raw',
        '/content/mucs_raw/train',
        '/content/mucs_raw',
        '/content'
    ]:
        candidate = os.path.join(cdir, fname)
        if os.path.exists(candidate):
            return candidate
    return apath

class MoLEWhisperPipeline:
    """
    PolyWhisper Mixture-of-LoRA-Experts Pipeline:
    - Base Whisper-medium backbone
    - Bengali Expert LoRA
    - English Expert LoRA
    - Trained Router MLP over decoder hidden states
    """
    def __init__(self, base_model_id, bn_adapter_path, en_adapter_path, router_pt_path=None, device='cuda'):
        self.device = device
        print('Loading Whisper processor ...')
        self.processor = WhisperProcessor.from_pretrained(base_model_id, language='Bengali', task='transcribe')
        
        print('Loading Base Whisper backbone ...')
        base = WhisperForConditionalGeneration.from_pretrained(
            base_model_id,
            torch_dtype=torch.float16 if device == 'cuda' else torch.float32
        )
        
        print('Mounting Bengali & English LoRA experts ...')
        self.model = PeftModel.from_pretrained(base, bn_adapter_path, adapter_name='bengali')
        self.model.load_adapter(en_adapter_path, adapter_name='english')
        
        # Combine experts into active weighted adapter (60% Bengali, 40% English)
        try:
            self.model.add_weighted_adapter(
                adapters=['bengali', 'english'],
                weights=[0.6, 0.4],
                adapter_name='mole_active',
                combination_type='linear'
            )
            self.model.set_adapter('mole_active')
            print('  Active adapter: mole_active (linear blend: 60% Bengali, 40% English)')
        except Exception as e:
            print('  Notice: fallback adapter set to bengali:', e)
            self.model.set_adapter('bengali')
            
        self.model.to(device).eval()
        
        self.router = None
        if router_pt_path and os.path.exists(router_pt_path):
            print('Loading MoLE Router weights ...')
            self.router = WhisperMoERouter(d_model=1024, hidden_dim=32, num_experts=2)
            self.router.load_state_dict(torch.load(router_pt_path, map_location=device))
            self.router.to(device).eval()
            print('✅ MoLE Pipeline ready with trained Router.')
        else:
            print('✅ MoLE Pipeline ready (Router analysis pending).')

    def transcribe_segments(self, df_records, batch_size=8):
        results = []
        routing_stats = {'bengali_tokens': 0, 'english_tokens': 0}
        
        for i in tqdm(range(0, len(df_records), batch_size), desc='  MoLE Transcription'):
            batch_rows = df_records.iloc[i:i + batch_size]
            waveforms = []
            for _, row in batch_rows.iterrows():
                apath = resolve_audio_path(str(row['audio_path']))
                s_sec = float(row.get('start_sec', 0.0))
                e_sec = float(row.get('end_sec', 0.0))
                dur   = (e_sec - s_sec) if e_sec > s_sec else None
                try:
                    wav, _ = librosa.load(apath, sr=16000, mono=True,
                                          offset=s_sec, duration=min(30.0, dur) if dur else 30.0)
                    if len(wav) == 0: wav = np.zeros(16000, dtype=np.float32)
                except Exception:
                    wav = np.zeros(16000, dtype=np.float32)
                waveforms.append(wav[:30 * 16000])
                
            inputs = self.processor(waveforms, sampling_rate=16000, return_tensors='pt', padding=True)
            input_features = inputs.input_features.to(self.device)
            if self.device == 'cuda':
                input_features = input_features.half()
                
            with torch.no_grad():
                predicted_ids = self.model.generate(
                    input_features,
                    language='Bengali',
                    task='transcribe',
                    max_new_tokens=448,
                    return_dict_in_generate=True,
                    output_hidden_states=True
                )
                
                # Analyze router allocations over generated token sequence
                if self.router is not None and hasattr(predicted_ids, 'decoder_hidden_states') and predicted_ids.decoder_hidden_states:
                    for step_hidden in predicted_ids.decoder_hidden_states:
                        h = step_hidden[-1]
                        r_logits = self.router(h.float())
                        chosen   = r_logits.argmax(dim=-1)
                        routing_stats['bengali_tokens'] += (chosen == 0).sum().item()
                        routing_stats['english_tokens'] += (chosen == 1).sum().item()
                
                decoded = self.processor.batch_decode(predicted_ids.sequences, skip_special_tokens=True)
                results.extend(decoded)
                
        return results, routing_stats

print('✅ MoLEWhisperPipeline class created with audio segment loading & router analysis')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Run Evaluation Benchmark (Baseline vs Path B MoLE)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, librosa
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE_MODEL_ID = 'SayedShaun/bengali-whisper-medium'

# Load validation set
df_val = pd.read_csv(VAL_CSV)
df_val = df_val[df_val['audio_path'].notna()].reset_index(drop=True)
if 'n_switches' in df_val.columns:
    df_val = df_val[df_val['n_switches'] >= 1].reset_index(drop=True)

# Subsample to 200 utterances if validation set is very large
if len(df_val) > 200:
    df_val = df_val.sample(n=200, random_state=42).reset_index(drop=True)

print(f'Evaluating benchmark on {len(df_val)} code-switched validation utterances ...')
references = [nfc(t) for t in df_val['transcript'].tolist()]
benchmark_results = {}

def score_predictions(hyps, name):
    score_wer = wer(references, [nfc(h) for h in hyps])
    total_ewer = EWERResult()
    for ref, hyp in zip(references, hyps):
        res = compute_ewer(ref, hyp)
        total_ewer.ref_english_words += res.ref_english_words
        total_ewer.emitted_correctly += res.emitted_correctly
        total_ewer.transliterated    += res.transliterated
        total_ewer.missing           += res.missing
    if total_ewer.ref_english_words > 0:
        total_ewer.ewer = total_ewer.emitted_correctly / total_ewer.ref_english_words
    return score_wer, total_ewer

# ── 1. BASELINE MODEL ─────────────────────────────────────
print('\n[1/2] Benchmarking Baseline Whisper ...')
proc_base = WhisperProcessor.from_pretrained(BASE_MODEL_ID, language='Bengali', task='transcribe')
model_base = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.float16 if device == 'cuda' else torch.float32
).to(device).eval()

base_hyps = []
for i in tqdm(range(0, len(df_val), 8), desc='  Baseline Transcription'):
    batch_rows = df_val.iloc[i:i+8]
    waves = []
    for _, r in batch_rows.iterrows():
        apath = resolve_audio_path(str(r['audio_path']))
        s_sec = float(r.get('start_sec', 0.0))
        e_sec = float(r.get('end_sec', 0.0))
        dur   = (e_sec - s_sec) if e_sec > s_sec else None
        try:
            w, _ = librosa.load(apath, sr=16000, mono=True, offset=s_sec, duration=min(30.0, dur) if dur else 30.0)
            if len(w) == 0: w = np.zeros(16000, dtype=np.float32)
        except Exception:
            w = np.zeros(16000, dtype=np.float32)
        waves.append(w[:30*16000])
        
    inps = proc_base(waves, sampling_rate=16000, return_tensors='pt', padding=True).input_features.to(device)
    if device == 'cuda': inps = inps.half()
    with torch.no_grad():
        out_ids = model_base.generate(inps, language='Bengali', task='transcribe', max_new_tokens=448)
    base_hyps.extend(proc_base.batch_decode(out_ids, skip_special_tokens=True))

base_wer, base_ewer = score_predictions(base_hyps, 'Baseline')
benchmark_results['Baseline'] = {'wer': base_wer, 'ewer': base_ewer.ewer, 'transliterated': base_ewer.transliterated}
del model_base; torch.cuda.empty_cache()

# ── 2. PATH B MoLE ROUTER ─────────────────────────────────
routing_stats = {'bengali_tokens': 0, 'english_tokens': 0}
if os.path.exists(BN_EXPERT + '/adapter_config.json') and os.path.exists(EN_EXPERT + '/adapter_config.json'):
    print('\n[2/2] Benchmarking Path B PolyWhisper MoLE Router ...')
    mole_pipeline = MoLEWhisperPipeline(
        base_model_id   = BASE_MODEL_ID,
        bn_adapter_path = BN_EXPERT,
        en_adapter_path = EN_EXPERT,
        router_pt_path  = ROUTER_WEIGHTS,
        device          = device
    )
    mole_hyps, routing_stats = mole_pipeline.transcribe_segments(df_val, batch_size=8)
    mole_wer, mole_ewer = score_predictions(mole_hyps, 'Path B')
    benchmark_results['Path B'] = {
        'wer': mole_wer,
        'ewer': mole_ewer.ewer,
        'transliterated': mole_ewer.transliterated,
        'routing_stats': routing_stats
    }
else:
    print('\n[2/2] Path B experts missing — run Notebooks B2 and B3 first.')
    benchmark_results['Path B'] = {'wer': None, 'ewer': None, 'transliterated': None}

# ── SUMMARY COMPARISON TABLE ──────────────────────────────
print('\n' + '=' * 72)
print('  PATH B EVALUATION RESULTS')
print('=' * 72)
print(f'{"Model":<25} | {"WER (↓ better)":<16} | {"EWER (↑ better)":<18} | {"Transliterated (⚠️)"}')
print('-' * 72)
for name, res in benchmark_results.items():
    if res['wer'] is not None:
        print(f"{name:<25} | {res['wer']*100:>12.2f}%    | {res['ewer']*100:>14.1f}%    | {res['transliterated']:>12}")
    else:
        print(f"{name:<25} | {'Pending':>14}    | {'Pending':>16}    | {'Pending':>12}")
print('=' * 72)

tot_routed = routing_stats['bengali_tokens'] + routing_stats['english_tokens']
if tot_routed > 0:
    bn_pct = routing_stats['bengali_tokens'] * 100 / tot_routed
    en_pct = routing_stats['english_tokens'] * 100 / tot_routed
    print(f'Path B Router Allocations: Bengali Expert: {bn_pct:.1f}% | English Expert: {en_pct:.1f}%')

# Save results JSON to Drive
out_json = DRIVE_ROOT + '/path_b_eval_results.json'
with open(out_json, 'w') as f:
    json.dump(benchmark_results, f, indent=2)
print(f'\n✅ Results saved to {out_json}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — Test on Real Hoichoi Video Files (.mp4)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, glob, os, json, librosa
import numpy as np
import torch

# Search candidate video clips across Drive and local Colab runtime
search_dirs = [DRIVE_ROOT, DRIVE_ROOT + '/videos', '/content', '/content/videos']
mp4_candidates = []
for sdir in search_dirs:
    if os.path.exists(sdir):
        mp4_candidates.extend(glob.glob(sdir + '/*.mp4'))
mp4_candidates = sorted(list(set(mp4_candidates)))

print(f'Found {len(mp4_candidates)} video(s):')
for f in mp4_candidates:
    print('  ▶', os.path.basename(f), f'({os.path.getsize(f)/1e6:.1f} MB)')

video_transcripts = {}

if not mp4_candidates:
    print('\n⚠️ No .mp4 files found yet!')
    print(f'   Upload video clips (bhojon_bilashi.mp4, feluda.mp4, etc.) to:')
    print(f'   {DRIVE_ROOT}/ or /content/ and re-run this cell.')
else:
    # Select pipeline to use for video transcription
    active_pipeline = mole_pipeline if 'mole_pipeline' in locals() else None
    
    for mp4_path in mp4_candidates:
        v_name = os.path.basename(mp4_path)
        out_wav = f'/content/{v_name}.wav'
        
        # Extract first 60 seconds at 16kHz mono
        subprocess.run([
            'ffmpeg', '-y', '-i', mp4_path, '-t', '60',
            '-ar', '16000', '-ac', '1', out_wav
        ], capture_output=True, check=True)
        
        wav, _ = librosa.load(out_wav, sr=16000, mono=True)
        chunk_size = 30 * 16000
        chunks = [wav[i:i+chunk_size] for i in range(0, len(wav), chunk_size) if len(wav[i:i+chunk_size]) > 1600]
        if not chunks: chunks = [wav]
        
        if active_pipeline is not None:
            inputs = active_pipeline.processor(chunks, sampling_rate=16000, return_tensors='pt', padding=True).input_features.to(device)
            if device == 'cuda': inputs = inputs.half()
            with torch.no_grad():
                pred_ids = active_pipeline.model.generate(inputs, language='Bengali', task='transcribe', max_new_tokens=448)
            decoded = active_pipeline.processor.batch_decode(pred_ids, skip_special_tokens=True)
        else:
            inputs = proc_base(chunks, sampling_rate=16000, return_tensors='pt', padding=True).input_features.to(device)
            if device == 'cuda': inputs = inputs.half()
            with torch.no_grad():
                pred_ids = model_base.generate(inputs, language='Bengali', task='transcribe', max_new_tokens=448)
            decoded = proc_base.batch_decode(out_ids, skip_special_tokens=True)
            
        full_text = ' '.join(decoded)
        words = full_text.split()
        latin_terms = [w for w in words if is_latin_word(w)]
        
        video_transcripts[v_name] = {
            'transcript': full_text,
            'latin_terms': latin_terms
        }
        
        print(f'\n── Video: {v_name} ' + '─' * 40)
        print(full_text[:300] + ('...' if len(full_text) > 300 else ''))
        print(f'  English / Code-Switched Terms: {latin_terms[:8]}')
        
    # Save transcripts to Drive
    with open(DRIVE_ROOT + '/video_transcripts_path_b.json', 'w', encoding='utf-8') as f:
        json.dump(video_transcripts, f, ensure_ascii=False, indent=2)
    print('\n✅ Video transcripts saved: video_transcripts_path_b.json')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7 — Contextual Ad Placement Engine (brands.json)
# Matches Video Scenes & Speech Contexts to Brand Creatives
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, os

# Locate brands catalog
brands_path = None
for candidate in [DRIVE_ROOT + '/brands.json', '/content/brands.json', 'brands.json']:
    if os.path.exists(candidate):
        brands_path = candidate
        break

if not brands_path:
    print('⚠️ brands.json not found! Please place brands.json in your Drive or Colab workspace.')
else:
    with open(brands_path, encoding='utf-8') as f:
        brands_catalog = json.load(f)
    print(f'✅ Loaded {len(brands_catalog)} brand profiles from {brands_path}\n')
    
    def match_ad_cues(transcript_text, brands):
        """
        Scores brand relevance against video transcript:
        - Positive context keyword matches (+1 each)
        - Negative safety filters (vetoes brand if negative context triggers)
        """
        t_lower = transcript_text.lower()
        scored_matches = []
        
        for b in brands:
            bid      = b['brand_id']
            bname    = b.get('display_name', bid)
            category = b.get('category', 'general')
            targets  = b.get('target_contexts', [])
            negatives = b.get('negative_contexts', [])
            
            # 1. Negative Safety Filter
            has_negative = any(neg.lower() in t_lower for neg in negatives)
            if has_negative:
                continue  # Safety violation: brand blocked for this scene
                
            # 2. Target Context Matching
            matched_keywords = [t for t in targets if t.lower() in t_lower]
            score = len(matched_keywords)
            
            if score > 0 or True:  # Include with baseline priority
                # Select optimal creative (default 15s or 20s Bengali ad)
                creatives = b.get('creatives', [])
                selected_creative = creatives[0] if creatives else {'id': f'{bid}_default', 'duration_sec': 15}
                
                scored_matches.append({
                    'brand_id': bid,
                    'brand_name': bname,
                    'category': category,
                    'score': score,
                    'matched_keywords': matched_keywords,
                    'recommended_creative': selected_creative
                })
                
        # Rank by score descending
        scored_matches.sort(key=lambda x: x['score'], reverse=True)
        return scored_matches
    
    ad_cue_sheet = {}
    for v_name, data in video_transcripts.items():
        text = data['transcript']
        matches = match_ad_cues(text, brands_catalog)
        top_match = matches[0] if matches else None
        
        ad_cue_sheet[v_name] = {
            'top_brand': top_match['brand_name'] if top_match else 'None',
            'category': top_match['category'] if top_match else 'None',
            'creative_id': top_match['recommended_creative']['id'] if top_match else 'None',
            'ad_duration_sec': top_match['recommended_creative'].get('duration_sec', 15) if top_match else 0,
            'matched_keywords': top_match['matched_keywords'] if top_match else [],
            'cue_timestamp_sec': 30.0  # Mid-roll break standard
        }
        
        print(f'📺 Video: {v_name}')
        if top_match:
            print(f'   ✦ Recommended Brand : {top_match["brand_name"]} ({top_match["category"]})')
            print(f'   ✦ Creative ID       : {top_match["recommended_creative"]["id"]} ({top_match["recommended_creative"].get("duration_sec")}s)')
            print(f'   ✦ Matched Keywords  : {top_match["matched_keywords"]}')
            print(f'   ✦ Insertion Cue     : Mid-roll at 00:30s\n')
            
    # Save Ad Cue Sheet to Drive
    cue_out = DRIVE_ROOT + '/contextual_ad_cue_sheet.json'
    with open(cue_out, 'w', encoding='utf-8') as f:
        json.dump(ad_cue_sheet, f, ensure_ascii=False, indent=2)
    print(f'✅ Contextual ad cue-sheet saved to: {cue_out}')